# Network traffic classification: beginner Colab notebook
We predict whether a network flow is **normal (0)** or **malicious (1)** using the UNSW-NB15 dataset. A row describes a flow; columns describe measurements such as duration, bytes and protocol.

Run cells from top to bottom. Use a **CPU runtime**; a GPU is not needed. This notebook keeps the four classifiers from the original project but explains each stage. No custom classes or training wrappers are needed.

The flow is: load data → prepare features → split training/validation → preprocess → train four models → compare validation scores → test the selected model → save results.

The dataset is not included in this repository. Obtain the prepared training and testing splits from the [official UNSW-NB15 project](https://research.unsw.edu.au/projects/unsw-nb15-dataset). Use CSV files if possible; Excel is also supported. Do not use the raw packet files for this notebook.

## 1. Install and import libraries
Pandas reads tables, scikit-learn prepares data and trains the first two models, and Matplotlib draws graphs. XGBoost and LightGBM provide the other two classifiers.

In [ ]:
%pip -q install "scikit-learn==1.6.1" "xgboost==2.1.4" "lightgbm==4.6.0" openpyxl joblib

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEED = 42


## 2. Choose how to load your files
Set `DATA_SOURCE` to `"upload"` for files on your computer, or `"drive"` for files already in Google Drive. For Drive, edit both paths below. CSV parsing is usually faster than Excel parsing; pandas reads the data after transfer and does not control upload speed. Keeping data in Drive avoids uploading it again every session. This cell copies Drive files to the Colab disk before reading.

For direct upload, select both files together. Their names must contain `training` and `testing`, respectively (for example, `UNSW_NB15_training-set.csv`).

In [ ]:
DATA_SOURCE = "upload"
TRAIN_PATH = "/content/drive/MyDrive/UNSW_NB15_training-set.csv"
TEST_PATH = "/content/drive/MyDrive/UNSW_NB15_testing-set.csv"

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    train_path = "/content/training" + Path(TRAIN_PATH).suffix
    test_path = "/content/testing" + Path(TEST_PATH).suffix
    shutil.copyfile(TRAIN_PATH, train_path)
    shutil.copyfile(TEST_PATH, test_path)
elif DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    names = list(uploaded)
    training_files = [name for name in names if "training" in name.lower()]
    testing_files = [name for name in names if "testing" in name.lower()]
    if len(training_files) != 1 or len(testing_files) != 1:
        raise ValueError("Select one training file and one testing file together.")
    train_path = training_files[0]
    test_path = testing_files[0]
    del uploaded
else:
    raise ValueError('DATA_SOURCE must be "upload" or "drive".')


In [ ]:
def read_table(path):
    if Path(path).suffix.lower() == ".csv":
        return pd.read_csv(path)
    if Path(path).suffix.lower() == ".xlsx":
        return pd.read_excel(path, engine="openpyxl")
    raise ValueError("Use a .csv or .xlsx file.")

train_data = read_table(train_path)
test_data = read_table(test_path)

for data in [train_data, test_data]:
    data.columns = data.columns.str.strip()
    if not data.columns.is_unique or "label" not in data.columns:
        raise ValueError("Each file needs unique column names and a label column.")
    data["label"] = pd.to_numeric(data["label"], errors="raise")
    if data["label"].isna().any() or set(data["label"].unique()) != {0, 1}:
        raise ValueError("Labels must contain both 0 (normal) and 1 (malicious).")
    data["label"] = data["label"].astype(int)

if set(train_data.columns) != set(test_data.columns):
    raise ValueError("Training and testing files must have the same columns.")
print("Training shape:", train_data.shape)
print("Testing shape:", test_data.shape)
display(train_data.head())


## 3. Understand the labels
The height of each bar is the number of training examples. Unequal class sizes can make accuracy misleading, so we compare models using malicious-class F1.

In [ ]:
train_data["label"].value_counts().sort_index().plot.bar()
plt.xticks([0, 1], ["Normal", "Malicious"], rotation=0)
plt.ylabel("Number of flows")
plt.title("Training class balance")
plt.show()


## 4. Separate inputs X from answers y
`X` contains measurements; `y` contains the answer we want to predict. Remove `label`, `attack_cat` and `id` from the inputs. `attack_cat` reveals the answer and would cause **data leakage**. An ID is just a row identifier.

Text columns such as protocol need encoding. Numeric columns may need missing-value replacement. We only identify types here; learned preprocessing happens after splitting.

In [ ]:
feature_columns = [column for column in train_data.columns
                   if column not in ["label", "attack_cat", "id"]]
X = train_data[feature_columns].copy()
y = train_data["label"].copy()
X_test = test_data[feature_columns].copy()
y_test = test_data["label"].copy()

categorical_columns = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_columns = [column for column in feature_columns if column not in categorical_columns]

for data in [X, X_test]:
    for column in categorical_columns:
        data[column] = data[column].fillna("Unknown").astype(str).str.strip()
    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")
        data[column] = data[column].replace([np.inf, -np.inf], np.nan)

print("Numeric columns:", numeric_columns)
print("Text columns:", categorical_columns)
print("Missing training values:", X.isna().sum().sum())


## 5. Create a validation set
The training portion teaches the model. The validation portion helps choose a model. The supplied test file is reserved for the final check.

Some flows have identical inputs. `hash_pandas_object` gives identical rows the same group identifier. `StratifiedGroupKFold` keeps those groups together while trying to preserve the normal/attack balance. We use one of five folds as validation (approximately 20%). This prevents duplicate rows appearing on both sides of the split.

In [ ]:
groups = pd.util.hash_pandas_object(X, index=False)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_indices, validation_indices = next(splitter.split(X, y, groups))

X_train = X.iloc[train_indices]
X_validation = X.iloc[validation_indices]
y_train = y.iloc[train_indices]
y_validation = y.iloc[validation_indices]

assert set(groups.iloc[train_indices]).isdisjoint(set(groups.iloc[validation_indices]))
if y_train.nunique() != 2 or y_validation.nunique() != 2:
    raise ValueError("Both split portions need both labels. Use the complete prepared dataset.")
print("Training rows:", len(X_train))
print("Validation rows:", len(X_validation))


## 6. Prepare numbers and text
- **Median imputation** replaces missing numbers with a median learned from training data.
- **Standard scaling** puts numbers on comparable scales for Logistic Regression.
- **One-hot encoding** turns categories into indicator columns. Unknown validation/test categories are allowed.

A `ColumnTransformer` applies the right steps to each column type. A `Pipeline` chains preprocessing and a classifier. Calling `.fit()` learns preprocessing only from the training portion. Sparse encoding avoids building a large dense table. Tree models do not need scaling; using the same scaled inputs here keeps the example simple.

In [ ]:
numeric_preparation = Pipeline([
    ("fill_missing", SimpleImputer(strategy="median", keep_empty_features=True)),
    ("scale", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("numbers", numeric_preparation, numeric_columns),
    ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categorical_columns)
], sparse_threshold=1.0)


## 7. Train Logistic Regression
Despite its name, Logistic Regression is a classifier. It learns a weighted combination of the inputs. `.fit(X_train, y_train)` learns the weights. `.predict()` returns 0 or 1. `max_iter` limits optimization iterations; it is not a neural-network epoch count. If a convergence warning appears, increase it before trusting this comparison.

In [ ]:
logistic_model = Pipeline([
    ("prepare", clone(preprocessor)),
    ("classify", LogisticRegression(max_iter=3000, random_state=SEED))
])
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_validation)
print("Validation F1:", f1_score(y_validation, logistic_predictions))


## 8. Train Random Forest
A decision tree asks questions about features. A random forest combines many trees. `n_estimators` is the number of trees; `max_depth` limits their size. These modest settings are a starting point for Colab CPU training.

In [ ]:
forest_model = Pipeline([
    ("prepare", clone(preprocessor)),
    ("classify", RandomForestClassifier(n_estimators=100, max_depth=20,
                                         min_samples_leaf=2, n_jobs=2, random_state=SEED))
])
forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_validation)
print("Validation F1:", f1_score(y_validation, forest_predictions))


## 9. Train XGBoost
Boosting adds trees sequentially to improve earlier predictions. `learning_rate` controls the size of each update. This uses CPU histogram training.

In [ ]:
xgboost_model = Pipeline([
    ("prepare", clone(preprocessor)),
    ("classify", XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1,
                               tree_method="hist", eval_metric="logloss", n_jobs=2,
                               random_state=SEED))
])
xgboost_model.fit(X_train, y_train)
xgboost_predictions = xgboost_model.predict(X_validation)
print("Validation F1:", f1_score(y_validation, xgboost_predictions))


## 10. Train LightGBM
LightGBM is another tree-boosting implementation. `num_leaves` controls tree complexity. This is a baseline comparison, not a full hyperparameter search.

In [ ]:
lightgbm_model = Pipeline([
    ("prepare", clone(preprocessor)),
    ("classify", LGBMClassifier(n_estimators=150, learning_rate=0.1, num_leaves=31,
                                verbosity=-1, n_jobs=2, random_state=SEED))
])
lightgbm_model.fit(X_train, y_train)
lightgbm_predictions = lightgbm_model.predict(X_validation)
print("Validation F1:", f1_score(y_validation, lightgbm_predictions))


## 11. Compare models and choose using validation F1
**Precision:** among predicted attacks, how many are attacks? **Recall:** among actual attacks, how many did we detect? **F1:** the harmonic mean of precision and recall. **Accuracy:** the fraction of all correct predictions.

All models use the same split. Only validation F1 chooses the winner. Change model settings above before running the final test; do not tune based on test results.

In [ ]:
models = {"Logistic Regression": logistic_model, "Random Forest": forest_model,
          "XGBoost": xgboost_model, "LightGBM": lightgbm_model}
rows = []
for name, model in models.items():
    predictions = model.predict(X_validation)
    rows.append({"Model": name,
                 "F1": f1_score(y_validation, predictions, zero_division=0),
                 "Precision": precision_score(y_validation, predictions, zero_division=0),
                 "Recall": recall_score(y_validation, predictions, zero_division=0),
                 "Accuracy": accuracy_score(y_validation, predictions)})

results = pd.DataFrame(rows).set_index("Model")
display(results.round(4))
best_name = results["F1"].idxmax()
print("Selected model:", best_name)
results.plot.bar(figsize=(10, 5), ylim=(0, 1.05))
plt.ylabel("Validation score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("validation_comparison.png")
plt.show()


## 12. Refit the winner and evaluate once on the test set
`clone` creates a fresh, unfitted copy with the same settings. We now fit it on all supplied training rows, including the previous validation portion, then evaluate on the supplied test set.

The confusion matrix shows correct predictions and mistakes: false positives are normal flows flagged as attacks; false negatives are attacks missed. This offline benchmark does not establish performance on live networks or unseen attack families.

In [ ]:
final_model = clone(models[best_name])
final_model.fit(X, y)
test_predictions = final_model.predict(X_test)
print("Selected model:", best_name)
print(classification_report(y_test, test_predictions, labels=[0, 1],
                            target_names=["Normal", "Malicious"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, labels=[0, 1],
                                        display_labels=["Normal", "Malicious"], cmap="Blues")
plt.tight_layout()
plt.savefig("test_confusion_matrix.png")
plt.show()

test_groups = pd.util.hash_pandas_object(X_test, index=False)
overlap = test_groups.isin(set(groups))
print("Test rows with identical predictors in training:", int(overlap.sum()))
if (~overlap).any() and y_test.loc[~overlap].nunique() == 2:
    print("F1 on test rows absent from training:",
          f1_score(y_test.loc[~overlap], test_predictions[~overlap.to_numpy()]))
else:
    print("No non-overlapping test subset with both classes is available.")


## 13. Save and download your work
The model file includes fitted preprocessing. The ZIP also contains validation scores, test predictions, and graphs. Download it before Colab disconnects. Only load joblib files you trust.

The prediction example below reuses the cleaned `X_test` table from section 4. For new files, repeat that same cleaning and supply the same feature columns.

In [ ]:
output = Path("beginner_results")
output.mkdir(exist_ok=True)
joblib.dump(final_model, output / "traffic_classifier.joblib")
results.to_csv(output / "validation_metrics.csv")
pd.DataFrame({"actual": y_test, "predicted": test_predictions}).to_csv(
    output / "test_predictions.csv", index=False)
for name in ["validation_comparison.png", "test_confusion_matrix.png"]:
    shutil.copyfile(name, output / name)
shutil.make_archive("beginner_results", "zip", output)
from google.colab import files
files.download("beginner_results.zip")


In [ ]:
loaded_model = joblib.load("beginner_results/traffic_classifier.joblib")
example_flows = X_test.head(5)
print("Predictions:", loaded_model.predict(example_flows))
print("Actual labels:", y_test.head(5).to_numpy())


## What to try next
Change one setting at a time (for example, Random Forest `n_estimators` from 100 to 200), compare validation scores, and record the experiment. There is no `epochs` parameter for these classical models. Keep the test set for a final locked experiment.

The original `UNSW_NB15_Classical_ML.ipynb` has more exploratory plots and evaluates all four refitted models. This notebook intentionally uses fewer plots and tests only the validation-selected winner. Its smaller model settings mean scores need not match the original.

Dataset credit: Nour Moustafa and Jill Slay, “UNSW-NB15: a comprehensive data set for network intrusion detection systems (UNSW-NB15 network data set),” MilCIS, 2015. See the official dataset page for usage terms. Scores are produced by your run; no benchmark results are claimed here.